# Question-Answering LLM from scratch (Colab)

A small GPT (~14M params) with **random initial weights** and a **tokenizer trained from scratch**.

Plan:
1. **Stage 1: pretrain** on TinyStories, so the model learns English.
2. **Stage 2: QA training** on SQuAD in the format `Context / Question / Answer`, with loss only on the answer tokens.

TinyStories has no questions, so a QA stage is needed. Both stages start from random weights, and no pretrained model is used.

**Runtime → Change runtime type → T4 GPU** before running.

#**Inizialize**

In [ ]:
!pip install -q datasets tokenizers

In [ ]:
import math, random, time, re, string, collections
import numpy as np
import torch, torch.nn as nn, torch.nn.functional as F
from datasets import load_dataset
from tokenizers import ByteLevelBPETokenizer

# ---------------- config ----------------
N_STORIES     = 100_000   # TinyStories used for stage 1 (more = better, slower)
VOCAB_SIZE    = 8192
BLOCK_SIZE    = 512
N_LAYER, N_HEAD, N_EMBD = 6, 6, 384
DROPOUT       = 0.1
BATCH_SIZE    = 32
PRETRAIN_STEPS = 5000     # stage 1
QA_STEPS       = 3000     # stage 2
PRETRAIN_LR, QA_LR = 6e-4, 2e-4
SEED = 1337

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
'''
This sets the starting point for the random number generators in Python, NumPy, and PyTorch.
Without it, weight initialization, data shuffling, and dropout would differ on every run, so you’d get slightly different results each time.
'''
device = "cuda" if torch.cuda.is_available() else "cpu"   #This uses the GPU if one is available and falls back to the CPU otherwise
print("device:", device)

## 1. Data

In [ ]:
#loads the dataset

dataset_name="roneneldan/TinyStories"
stories_train = load_dataset(dataset_name, split=f"train[:{N_STORIES}]")
#This loads only the first N_STORIES examples from the training split. The slice syntax train[:N] is how datasets lets you take a subset.
#This slicing help to save space and do training in ouly some data

stories_val   = load_dataset(dataset_name, split="validation[:2000]")
#This takes the first 2,000 validation stories.
#These are held out from training and used to measure loss on text the model hasn’t seen, which tells you whether it’s learning or just memorizing.

squad = load_dataset("rajpurkar/squad")
#This loads SQuAD, a question-answering dataset where each example has a context paragraph, a question, and an answer span

In [ ]:
#look at the data
print(stories_train[0]["text"][:300])
print(squad["train"][0])

In [ ]:
# Keep SQuAD examples short enough for a small model

'''
it does two things:
Throws away the hard examples: SQuAD has long paragraphs and long answers that a tiny model can’t handle. The cell keeps only examples where the paragraph is 700 characters
                               or less and the answer is 60 characters or less, then prints how many are left.
Defines the question format: make_prompt turns an example into text like this:
                             Context: <the paragraph>
                             Question: <the question>
                             Answer:<the answer>
During training, the correct answer gets added after Answer so the model learns to complete it.
At test time, you stop at Answer and let the model write the answer itself.
'''


def keep(ex):
    return len(ex["context"]) <= 700 and len(ex["answers"]["text"][0]) <= 60

qa_train = squad["train"].filter(keep)
qa_val   = squad["validation"].filter(keep)
print("QA train / val:", len(qa_train), len(qa_val))

def make_prompt(context, question):
    return f"Context: {context}\nQuestion: {question}\nAnswer:"

## 2. Tokenizer (trained from scratch)

In [ ]:
EOS = "<|endoftext|>"

def text_iter():
    for i in range(0, len(stories_train), 1000):
        yield from stories_train[i:i+1000]["text"]
    for ex in qa_train:
        yield make_prompt(ex["context"], ex["question"]) + " " + ex["answers"]["text"][0]

tok = ByteLevelBPETokenizer()
tok.train_from_iterator(text_iter(), vocab_size=VOCAB_SIZE, min_frequency=2, special_tokens=[EOS])
EOS_ID = tok.token_to_id(EOS)
tok.save("tokenizer.json")
print("vocab:", tok.get_vocab_size(), "| eos id:", EOS_ID)
print(tok.encode("Once upon a time, there was a cat.").tokens)

In [ ]:
# ---- Stage 1 data: one long token stream, documents separated by EOS ----
def tokenize_stream(texts, chunk=2000):
    out = []
    for i in range(0, len(texts), chunk):
        for enc in tok.encode_batch(texts[i:i+chunk]):
            out.extend(enc.ids); out.append(EOS_ID)
    return np.array(out, dtype=np.uint16)

train_stream = tokenize_stream(stories_train["text"])
val_stream   = tokenize_stream(stories_val["text"])
print("train tokens:", len(train_stream), "| val tokens:", len(val_stream))

def get_batch_lm(split="train"):
    data = train_stream if split == "train" else val_stream
    ix = np.random.randint(0, len(data) - BLOCK_SIZE - 1, BATCH_SIZE)
    x = np.stack([data[i:i+BLOCK_SIZE] for i in ix]).astype(np.int64)
    y = np.stack([data[i+1:i+1+BLOCK_SIZE] for i in ix]).astype(np.int64)
    return torch.from_numpy(x).to(device), torch.from_numpy(y).to(device)

In [ ]:
# ---- Stage 2 data: prompt tokens are masked (-100), loss only on answer + EOS ----
def build_qa(ds):
    prompts = [make_prompt(e["context"], e["question"]) for e in ds]
    answers = [" " + e["answers"]["text"][0] for e in ds]
    p_ids = [e.ids for e in tok.encode_batch(prompts)]
    a_ids = [e.ids + [EOS_ID] for e in tok.encode_batch(answers)]
    examples = []
    for p, a in zip(p_ids, a_ids):
        ids = p + a
        if len(ids) > BLOCK_SIZE + 1:
            continue
        labels = [-100] * len(p) + a
        examples.append((ids, labels))
    return examples

qa_train_tok = build_qa(qa_train)
qa_val_tok   = build_qa(qa_val)
print("usable QA examples:", len(qa_train_tok), len(qa_val_tok))

def get_batch_qa(split="train"):
    pool = qa_train_tok if split == "train" else qa_val_tok
    batch = random.sample(pool, BATCH_SIZE)
    L = max(len(ids) for ids, _ in batch) - 1
    x = torch.zeros(BATCH_SIZE, L, dtype=torch.long)
    y = torch.full((BATCH_SIZE, L), -100, dtype=torch.long)
    for i, (ids, labels) in enumerate(batch):
        n = len(ids) - 1
        x[i, :n] = torch.tensor(ids[:-1])
        y[i, :n] = torch.tensor(labels[1:])
    return x.to(device), y.to(device)

## 3. Model architecture (GPT-style decoder)

In [ ]:
#defines the model itself: a small GPT-style transformer (decoder-only, like GPT-2)

class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.qkv = nn.Linear(N_EMBD, 3 * N_EMBD, bias=False)
        self.proj = nn.Linear(N_EMBD, N_EMBD, bias=False)
        self.drop = DROPOUT
    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).split(C, dim=2)
        q, k, v = [t.view(B, T, N_HEAD, C // N_HEAD).transpose(1, 2) for t in (q, k, v)]
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True,
                                           dropout_p=self.drop if self.training else 0.0)
        return self.proj(y.transpose(1, 2).contiguous().view(B, T, C))

class Block(nn.Module):
    def __init__(self):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(N_EMBD), nn.LayerNorm(N_EMBD)
        self.attn = Attention()
        self.mlp = nn.Sequential(nn.Linear(N_EMBD, 4 * N_EMBD, bias=False), nn.GELU(),
                                 nn.Linear(4 * N_EMBD, N_EMBD, bias=False))
        self.drop = nn.Dropout(DROPOUT)
    def forward(self, x):
        x = x + self.drop(self.attn(self.ln1(x)))
        return x + self.drop(self.mlp(self.ln2(x)))

class GPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.tok_emb = nn.Embedding(vocab_size, N_EMBD)
        self.pos_emb = nn.Embedding(BLOCK_SIZE, N_EMBD)
        self.drop = nn.Dropout(DROPOUT)
        self.blocks = nn.ModuleList([Block() for _ in range(N_LAYER)])
        self.ln_f = nn.LayerNorm(N_EMBD)
        self.head = nn.Linear(N_EMBD, vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight          # weight tying
        self.apply(self._init)
        for n, p in self.named_parameters():            # scaled init for residual projections
            if n.endswith("proj.weight") or n.endswith("mlp.2.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * N_LAYER))
    def _init(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)
    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.tok_emb(idx) + self.pos_emb(torch.arange(T, device=idx.device)))
        for b in self.blocks: x = b(x)
        logits = self.head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens=64, temperature=0.0, top_k=40, stop_id=None):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -BLOCK_SIZE:])
            logits = logits[:, -1, :].float()
            if temperature == 0:
                nxt = logits.argmax(-1, keepdim=True)
            else:
                logits = logits / temperature
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = -float("inf")
                nxt = torch.multinomial(F.softmax(logits, -1), 1)
            idx = torch.cat([idx, nxt], dim=1)
            if stop_id is not None and nxt.item() == stop_id: break
        return idx

model = GPT(tok.get_vocab_size()).to(device)
print(f"{sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

## 4. Defines the training machinery

In [ ]:
def lr_at(step, total, base_lr, warmup=200):
    if step < warmup: return base_lr * (step + 1) / warmup
    prog = (step - warmup) / max(1, total - warmup)
    return base_lr * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * prog)))

@torch.no_grad()
def eval_loss(get_batch, iters=20):
    model.eval(); losses = []
    for _ in range(iters):
        x, y = get_batch("val")
        with torch.autocast(device, dtype=amp_dtype, enabled=device == "cuda"):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train(); return sum(losses) / len(losses)

def train(get_batch, steps, base_lr, eval_every=500, name=""):
    opt = torch.optim.AdamW(model.parameters(), lr=base_lr, betas=(0.9, 0.95), weight_decay=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda" and amp_dtype == torch.float16))
    model.train(); t0 = time.time()
    for step in range(steps):
        for g in opt.param_groups: g["lr"] = lr_at(step, steps, base_lr)
        x, y = get_batch("train")
        with torch.autocast(device, dtype=amp_dtype, enabled=device == "cuda"):
            _, loss = model(x, y)
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt); scaler.update()
        if step % eval_every == 0 or step == steps - 1:
            print(f"[{name}] step {step:5d}/{steps} | train {loss.item():.3f} | "
                  f"val {eval_loss(get_batch):.3f} | {time.time()-t0:.0f}s")

## 5. Stage 1: pretrain on TinyStories

In [ ]:
train(get_batch_lm, PRETRAIN_STEPS, PRETRAIN_LR, name="stories")
torch.save(model.state_dict(), "stage1_stories.pt")

In [ ]:
# Sanity check: should write simple, mostly coherent children's stories
ids = torch.tensor([tok.encode("Once upon a time, there was a little").ids], device=device)
out = model.generate(ids, max_new_tokens=100, temperature=0.8, stop_id=EOS_ID)
print(tok.decode(out[0].tolist()))

## 6. Stage 2: teach the Question → Answer format (SQuAD)
TinyStories has no questions, so we derive them with simple templates. Because the answers come straight from each story, the model must learn to read the story and answer from it (not just memorise).

In [ ]:
train(get_batch_qa, QA_STEPS, QA_LR, eval_every=250, name="qa")
torch.save(model.state_dict(), "stage2_qa.pt")

## 7. Ask questions

In [ ]:
def ask(context, question, max_new_tokens=40):
    ids = torch.tensor([tok.encode(make_prompt(context, question)).ids], device=device)
    with torch.autocast(device, dtype=amp_dtype, enabled=device == "cuda"):
        out = model.generate(ids, max_new_tokens=max_new_tokens, temperature=0.0, stop_id=EOS_ID)
    new = out[0, ids.shape[1]:].tolist()
    if new and new[-1] == EOS_ID: new = new[:-1]
    return tok.decode(new).strip()

description = "Tom has a red ball. He plays with it in the park every morning. His dog, Max, likes to chase the ball."
print(ask(description, "What color is Tom's ball?"))
print(ask(description, "What is the name of Tom's dog?"))
print(ask(description, "Where does Tom play?"))

In [ ]:
# Quick quantitative check on SQuAD validation (exact match and token F1)
def norm(s):
    s = s.lower()
    s = "".join(ch for ch in s if ch not in set(string.punctuation))
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def f1(pred, gold):
    p, g = norm(pred).split(), norm(gold).split()
    common = collections.Counter(p) & collections.Counter(g)
    n = sum(common.values())
    if n == 0: return 0.0
    pr, rc = n / len(p), n / len(g)
    return 2 * pr * rc / (pr + rc)

N = 200; em = f = 0
for ex in qa_val.select(range(N)):
    pred = ask(ex["context"], ex["question"])
    golds = ex["answers"]["text"]
    em += max(norm(pred) == norm(g) for g in golds)
    f  += max(f1(pred, g) for g in golds)
print(f"EM: {100*em/N:.1f} | F1: {100*f/N:.1f}  (on {N} val examples)")

## 8. (Optional) Save to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs("/content/drive/MyDrive/tinystories_qa", exist_ok=True)
!cp pretrained.pt qa_model.pt tokenizer.json /content/drive/MyDrive/tinystories_qa/